Use before everthing

In [ ]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Advanced pick up

In [ ]:
import time

def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """ Drives toward or away from an AprilTag to hit the perfect distance. """
    print("Adjusting position...")
    deadzone = 0.02  # Prevents the robot from moving back and forth constantly
    
    while True:
        AP_info = got.get_apriltag_total_info()
        if not AP_info or len(AP_info) == 0:
            got.mecanum_stop()
            time.sleep(0.1) 
            continue 

        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]
        
        # Centering Logic (Strafe)
        current_strafe = 0
        if AP_x < 320 - gap:
            current_strafe = -strafe_spd
        elif AP_x > 320 + gap:
            current_strafe = strafe_spd

        # Distance Logic (Forward/Backward)
        if AP_distance > (distance + deadzone):
            # Too far: Move Forward
            got.mecanum_move_xyz(current_strafe, fwd_spd, 0)
        elif AP_distance < (distance - deadzone):
            # Too close: Move Backward (Negative fwd_spd)
            got.mecanum_move_xyz(current_strafe, -fwd_spd, 0)
        else:
            # Within the "Sweet Spot": Stop
            got.mecanum_stop()
            print("Perfect distance reached.")
            time.sleep(1.0)
            break

def pick_up():
    """ Executes pickup sequence. """
    AP_info = got.get_apriltag_total_info()
    if not AP_info:
        print("Error: Tag disappeared!")
        return

    # 1. Prepare Arm & Open
    got.mechanical_joint_control(0, 30, 30, 1000)
    got.mechanical_clamp_release()
    time.sleep(1.5)

    # 2. Reach and Grab
    # Using your calculation logic
    joint1 = int(max(min((AP_info[0][1] - 320) * -0.1, 90), -90))
    joint3 = int(max(min(AP_info[0][6] * 100 - 80, 90), -90))
    
    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1.2)
    got.mechanical_clamp_close()
    time.sleep(1.5)

    # 3. Retract to carry position
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("Picked up!")
    print("Pickup complete.")

def put_down():
    """
    Executes the release sequence to safely place the object.
    """
    print("Starting Put Down sequence...")
    
    # 1. Lower the arm to the placement position (Adjust Joint 3 for height)
    # We use a similar angle to the pickup so it reaches the floor/table
    time.sleep(1.5)
    got.mechanical_single_joint_control(joint=2, angle=-20, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-20, duration=500)
    
    
    time.sleep(1.5)

    # 2. Release the object
    got.mechanical_clamp_release()
    print("Object released.")
    time.sleep(1.0)

    print("Put down")
    print("Put down complete.")
    


try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # Now handles being too close OR too far automatically
            AP_centralization_approaching(distance=0.13, gap=20, fwd_spd=10, strafe_spd=7)
            
            # Execute the grab
            pick_up()
            
            # Put down the object
            put_down()
            
            # Wait and then reset arm to carry/safety position
            time.sleep(2)
            # Simplified reset using your joint control
            got.mechanical_joint_control(0, 90, 90, 1000)
            got.mechanical_clamp_close() 
           
            break  # Exit after one successful cycle

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")

Basic Pick up

In [ ]:
import time

def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drives toward an AprilTag with safety checks to prevent IndexErrors.
    """
    print("Approaching target...")
    
    while True:
        AP_info = got.get_apriltag_total_info()
        
        # ERROR FIX 1: Check if the list is empty before accessing index [0]
        if not AP_info or len(AP_info) == 0:
            print("Tag lost! Searching...")
            got.mecanum_stop()
            time.sleep(0.1) 
            continue # Skip to the next loop iteration to try and find it again

        # Safely extract data now that we know it exists
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # ERROR FIX 2: Combined movement for smoother pathing
        current_strafe = 0
        if AP_x < 320 - gap:
            current_strafe = -strafe_spd
        elif AP_x > 320 + gap:
            current_strafe = strafe_spd

        if AP_distance > distance:
            # Move forward and strafe at the same time
            got.mecanum_move_xyz(current_strafe, fwd_spd, 0)
        else:
            # We reached the goal
            got.mecanum_stop()
            print("Target reached. Stabilizing...")
            time.sleep(1.0) # ERROR FIX 3: Let inertia settle before picking up
            break

def pick_up():
    """
    Executes pickup sequence with validation logic.
    """
    AP_info = got.get_apriltag_total_info()
    
    # Validation: Ensure tag is still there before moving arm
    if not AP_info:
        print("Error: Tag disappeared right before pickup!")
        return

    AP_x = AP_info[0][1]
    AP_distance = AP_info[0][6]

    # 1. Prepare Arm
    # Using a safe mid-point for Joint 2 (30) so it doesn't hit the floor
    got.mechanical_joint_control(0, 30, 30, 1000)
    got.mechanical_clamp_release()
    time.sleep(1.5)

    # 2. Calculate Pose
    # Added 'max/min' clipping to keep values in safe mechanical ranges
    joint1 = int(max(min((AP_x - 320) * -0.1, 90), -90))
    joint3 = int(max(min(AP_distance * 100 - 80, 90), -90))

    # 3. Reach and Grab
    print(f"Executing Grab -> J1: {joint1}, J3: {joint3}")
    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1.2)
    
    got.mechanical_clamp_close()
    time.sleep(1.5)

    # 4. Retract to safe carry position
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("Pickup complete.")

# Main Execution
try:
    AP_centralization_approaching()
    pick_up()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    got.mecanum_stop()


try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # A tag is visible — approach it and pick up the object.
            # Lower speeds (fwd_spd=5, strafe_spd=7) for more precise final approach.
            AP_centralization_approaching(distance= 1, gap=20, fwd_spd=5, strafe_spd=7)
            pick_up()

            print("Picked up!")
            break  # Exit after one successful pick-and-place

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

Reset

In [ ]:
got.mechanical_joint_control(angle1=0, angle2=90, angle3=90, duration=500)
got.mechanical_clamp_close() # Close the gripper



All functions so far

In [ ]:
#  Gripper control 
got.mechanical_clamp_release() # Open the gripper
got.mechanical_clamp_close() # Close the gripper

#  Arm reset 
# Returns all arm joints to their default/home position.
got.mechanical_arms_restory()

#  Move all arm joints simultaneously 
# angle1/2/3 are target angles (degrees); duration is movement time in ms.
got.mechanical_joint_control(angle1=0, angle2=0, angle3=0, duration=500)

#  Move a single arm joint 
# joint: 1=base, 2=middle, 3=furthest; angle in degrees; duration in ms.
got.mechanical_single_joint_control(joint=1, angle=0, duration=500)